# Quaternions and spin — the same four numbers

**The punchline.** A qubit rotation **is** a unit quaternion. Not "is analogous to", not
"is a cousin of" — the same four numbers, the same multiplication table, the same strange
half-angles. And the $-1$ that a quaternion carries silently after a $360°$ turn is
something a qubit can actually *measure*.

Quaternions were invented in 1843 for 3D geometry, eighty years before anyone had heard of
spin. Half-angles were their most-complained-about feature: to turn a vector by $\theta$ you
build a quaternion out of $\theta/2$. Textbooks call this a "quirk of the parametrisation".
It is not a quirk — it is a **pair of mirrors**, and two mirrors set $\theta/2$ apart turn
the world by $\theta$. It is also the same fact that makes an electron need two full turns
to come back to itself, and this notebook is going to identify the two exactly, numerically,
on qsim's own gates.

Background you may want first:

- **[01 — States and gates](../01-states-and-gates.ipynb)** introduces the qubit, Dirac
  notation and the Bloch sphere from scratch.
- **[one_qubit_playground](one_qubit_playground.ipynb)** closes on the observation that
  $R_x(2\pi) = -I$ — a full turn multiplies the state by $-1$. That cell is where this
  notebook starts, and Part 6 here is the promised answer to "so is that sign *real*?"

Everything else is built from scratch, in plain NumPy, in this notebook. The route:

1. quaternions from nothing — the multiplication table, and rotating 3-vectors by sandwiching;
2. where the half-angle actually comes from: a rotation is two reflections, and a quaternion
   is the product of its two mirror normals;
3. the dictionary between quaternions and one-qubit gates, checked on qsim's own $R_x$,
   $R_y$, $R_z$ — components *and* products;
4. the same rotation run twice in parallel, once as a quantum state and once as three real
   numbers pushed around by quaternion algebra;
5. the double cover: sweep to $4\pi$ and watch the two descriptions come apart;
6. the finale, where the $-1$ stops being bookkeeping and becomes a measurement outcome.

## Part 1 — Quaternions from nothing, in plain NumPy

A **quaternion** is four real numbers written as

$$q = a + b\,\mathbf{i} + c\,\mathbf{j} + d\,\mathbf{k},$$

exactly the way a complex number is two real numbers written as $a + b\,\mathbf{i}$.
Addition is componentwise and boring. Multiplication is the whole story, and it is fixed by
one line Hamilton famously carved into a Dublin bridge in 1843:

$$\mathbf{i}^2 = \mathbf{j}^2 = \mathbf{k}^2 = \mathbf{i}\mathbf{j}\mathbf{k} = -1.$$

Everything else follows from expanding that. In particular $\mathbf{ij} = \mathbf{k}$,
$\mathbf{jk} = \mathbf{i}$, $\mathbf{ki} = \mathbf{j}$ — cyclic, like a right-handed
coordinate frame — while going *backwards* costs a minus sign: $\mathbf{ji} = -\mathbf{k}$.
So quaternion multiplication is **not commutative**. That is not a defect; it is the point.
Rotations in 3D do not commute either (turn a book about $x$ then $y$, then try the other
order), so anything that represents rotations faithfully had better not commute.

Two more definitions we will need:

- the **conjugate** $\bar q = a - b\mathbf{i} - c\mathbf{j} - d\mathbf{k}$, which flips the
  three "vector" components and leaves the "scalar" component $a$ alone;
- the **norm** $\lVert q\rVert = \sqrt{a^2+b^2+c^2+d^2}$, the ordinary length of the
  4-vector. A quaternion with norm 1 is a **unit quaternion**, and those are the ones that
  represent rotations.

We store a quaternion as a plain length-4 NumPy array `[a, b, c, d]`. No classes, no
operator overloading — the arithmetic should stay visible.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import qsim
from qsim import Circuit
from qsim.gates import H, Rx, Ry, Rz, X, Y, Z


def hamilton(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    """The quaternion product p*q, for quaternions stored as [a, b, c, d]."""
    a1, b1, c1, d1 = p
    a2, b2, c2, d2 = q
    # Expand (a1 + b1 i + c1 j + d1 k)(a2 + b2 i + c2 j + d2 k) term by term and collect,
    # using nothing but i^2 = j^2 = k^2 = ijk = -1 and its consequences:
    #     i*j =  k,   j*k =  i,   k*i =  j       (cyclic: the right-handed order)
    #     j*i = -k,   k*j = -i,   i*k = -j       (backwards costs a minus sign)
    # The scalar row collects the three squares, which is where its minus signs come from.
    # Each vector row collects one "forwards" pair (+) and one "backwards" pair (-), which
    # is exactly the pattern of a cross product — see the markdown below.
    return np.array([
        a1 * a2 - b1 * b2 - c1 * c2 - d1 * d2,
        a1 * b2 + b1 * a2 + c1 * d2 - d1 * c2,
        a1 * c2 - b1 * d2 + c1 * a2 + d1 * b2,
        a1 * d2 + b1 * c2 - c1 * b2 + d1 * a2,
    ])


def conjugate(q: np.ndarray) -> np.ndarray:
    """The conjugate qbar: negate the i, j, k parts, keep the scalar part."""
    # Elementwise multiply by [1, -1, -1, -1]. For a *unit* quaternion the conjugate is
    # also the inverse, since q * qbar = ||q||^2 = 1.
    return q * np.array([1.0, -1.0, -1.0, -1.0])


def quat_norm(q: np.ndarray) -> float:
    """The length of the 4-vector, sqrt(a^2 + b^2 + c^2 + d^2)."""
    return float(np.sqrt(q @ q))  # q @ q is the dot product of q with itself

Before trusting `hamilton`, make it recite the multiplication table. We build the four basis
quaternions $1, \mathbf{i}, \mathbf{j}, \mathbf{k}$ as the four standard basis vectors and
multiply them together.

In [ ]:
one = np.array([1.0, 0.0, 0.0, 0.0])
qi = np.array([0.0, 1.0, 0.0, 0.0])
qj = np.array([0.0, 0.0, 1.0, 0.0])
qk = np.array([0.0, 0.0, 0.0, 1.0])

print("i*i      =", hamilton(qi, qi), "  (should be -1)")
print("j*j      =", hamilton(qj, qj), "  (should be -1)")
print("k*k      =", hamilton(qk, qk), "  (should be -1)")
print("i*j*k    =", hamilton(hamilton(qi, qj), qk), "  (should be -1)")
print()
print("i*j      =", hamilton(qi, qj), "  ( = +k)")
print("j*i      =", hamilton(qj, qi), "  ( = -k, the other way round)")
print()
print("1 is the identity:", np.allclose(hamilton(one, qj), qj))

The table checks out, and the non-commutativity is right there in the two middle lines:
$\mathbf{ij}$ and $\mathbf{ji}$ differ by a sign.

### Why unit quaternions rotate vectors — the rule, stated

Here is the trick that made quaternions famous, stated first and explained afterwards (Part 2
is the explanation; this section is the rule and a numerical check that it works).

Take a 3-vector $\vec v = (v_x, v_y, v_z)$ and embed it as a **pure quaternion** — scalar
part zero, vector part $\vec v$:

$$v = 0 + v_x\mathbf{i} + v_y\mathbf{j} + v_z\mathbf{k}.$$

Take a unit axis $\hat n$ and an angle $\theta$, and build

$$q \;=\; \cos\tfrac{\theta}{2} \;+\; \sin\tfrac{\theta}{2}\,
   \bigl(n_x\mathbf{i} + n_y\mathbf{j} + n_z\mathbf{k}\bigr).$$

Then the **sandwich product**

$$v' \;=\; q\,v\,\bar q$$

comes out pure again (its scalar part is exactly zero), and its vector part is $\vec v$
rotated about $\hat n$ by $\theta$, right-hand rule.

Two things to notice before we test it.

**The half-angle is already here.** No quantum mechanics has been mentioned. This is 1843
geometry, and $\theta/2$ is already in the formula. Something about the sandwich is
responsible — $q$ appears twice, so whatever angle it carries gets applied twice — but that
is a hint, not a reason, and Part 2 turns it into one.

**Therefore $q$ and $-q$ do the same thing.** Negating $q$ negates $\bar q$ too, and the two
minus signs cancel inside the sandwich. So *two* unit quaternions describe every rotation.
Hold onto that; it is the whole of Part 5, and Part 2 will say what the two signs *are*.

The ground truth we will check against is the ordinary axis–angle rotation matrix, built by
hand from Rodrigues' formula

$$R = \cos\theta\, \mathbb{1} \;+\; \sin\theta\, K \;+\; (1 - \cos\theta)\, \hat n \hat n^{\mathsf T},$$

where $K$ is the matrix that does "cross product with $\hat n$".

In [ ]:
def axis_angle_quaternion(axis: np.ndarray, theta: float) -> np.ndarray:
    """The unit quaternion for a rotation by theta about `axis` (need not be normalised)."""
    unit = axis / np.linalg.norm(axis)
    # Scalar part cos(theta/2); vector part sin(theta/2) times the axis. The * unpacking
    # splices the three components of the array into the four-element literal.
    return np.array([np.cos(theta / 2.0), *(np.sin(theta / 2.0) * unit)])


def rotate_by_quaternion(q: np.ndarray, v: np.ndarray) -> np.ndarray:
    """Rotate the 3-vector v by the unit quaternion q, via the sandwich q v qbar."""
    pure = np.array([0.0, v[0], v[1], v[2]])          # embed v as a pure quaternion
    out = hamilton(hamilton(q, pure), conjugate(q))
    return out[1:]                                     # scalar part comes back 0; drop it


def axis_angle_matrix(axis: np.ndarray, theta: float) -> np.ndarray:
    """The 3x3 rotation matrix for the same rotation, from Rodrigues' formula."""
    n = axis / np.linalg.norm(axis)
    # K is the cross-product matrix of n: K @ v equals np.cross(n, v) for every v.
    k = np.array([[0.0, -n[2], n[1]], [n[2], 0.0, -n[0]], [-n[1], n[0], 0.0]])
    # np.outer(n, n) is the 3x3 matrix n n^T, whose action on v is n * (n . v) --
    # the projection of v onto the axis, which a rotation about that axis leaves alone.
    return (
        np.cos(theta) * np.eye(3)
        + np.sin(theta) * k
        + (1.0 - np.cos(theta)) * np.outer(n, n)
    )


rot_rng = np.random.default_rng(1843)
worst = 0.0
print(f"{'axis':>22} {'theta':>7}   {'q v qbar':>26}   {'R v':>26}")
for _ in range(4):
    axis = rot_rng.normal(size=3)
    theta = rot_rng.uniform(0.0, 2.0 * np.pi)
    v = rot_rng.normal(size=3)
    by_quaternion = rotate_by_quaternion(axis_angle_quaternion(axis, theta), v)
    by_matrix = axis_angle_matrix(axis, theta) @ v
    worst = max(worst, float(np.abs(by_quaternion - by_matrix).max()))
    unit_axis = axis / np.linalg.norm(axis)
    print(f"{np.array2string(unit_axis, precision=2):>22} {theta:7.3f}   "
          f"{np.array2string(by_quaternion, precision=3):>26}   "
          f"{np.array2string(by_matrix, precision=3):>26}")

sandwich_error = worst
print(f"\nlargest disagreement over the four trials: {sandwich_error:.2e}")

Agreement to machine precision. The sandwich really is a rotation, and building it out of
$\theta/2$ really does produce a turn by $\theta$.

That is the entire quaternion crash course. Four numbers, one multiplication rule, one
sandwich. **Nothing above knows what a qubit is.**

## Part 2 — Where the half-angle comes from: mirrors

Part 1 handed you two things and asked you to take them on trust: the sandwich
$q\,v\,\bar q$, and the instruction to build $q$ out of $\theta/2$. The excuse offered for
the half — "$q$ appears twice, so the angle gets applied twice" — has the right shape, but
it is not an argument. It does not say why a rotation should be a product of *two* of
anything, and if you push on it you find you cannot even say what a single $q$ does to a
vector on its own.

There is a real answer, and it is older than quaternions. It is this:

> **A rotation is two reflections.**

Reflections are the easy part of 3D geometry — you can build one in your head, and the
formula for one is two lines of NumPy. Two mirrors set at angle $\alpha$ compose to a
rotation by $2\alpha$; that is the fact a kaleidoscope runs on. And in quaternion language a
single mirror turns out to be the simplest sandwich there is. Once both halves are on the
table, $q\,v\,\bar q$ stops being a trick: **$q$ is the pair of mirrors**, and the
half-angle is nothing but the angle between them.

Three steps: the 2D picture, where you can see the doubling; 3D reflections in plain NumPy,
checked numerically; and then the observation that collapses the whole thing.

*(Nothing in this part is quantum. The coda at the end asks which gates are mirrors, and the
answer is a nice surprise.)*

### Two mirrors in the plane

Start in 2D, where you can draw the whole thing.

A mirror in the plane is a **line through the origin**. Take the line at angle $\theta_1$,
with unit direction $\hat u = (\cos\theta_1, \sin\theta_1)$. Reflecting a vector in it keeps
the part along the line and flips the part across it:

$$M_1\vec v \;=\; 2(\vec v\cdot\hat u)\,\hat u \;-\; \vec v .$$

(Notice that even this humble formula uses $\hat u$ twice. Mirrors are sandwiches from the
start.)

Now do it with bearings instead of components, which makes the answer obvious. Reflection
in a line at $\theta_1$ *reverses* the angle you measure from that line: a vector at bearing
$\beta$ sits $\beta - \theta_1$ past the mirror, so its image sits $\beta - \theta_1$ before
it, at bearing $2\theta_1 - \beta$. Reflect that in the second mirror at $\theta_2$:

$$\beta \;\xrightarrow{\;M_1\;}\; 2\theta_1 - \beta
      \;\xrightarrow{\;M_2\;}\; 2\theta_2 - (2\theta_1 - \beta)
      \;=\; \beta + 2(\theta_2 - \theta_1).$$

Two facts fall out of that one line, and both matter.

- The result does not depend on $\beta$ at all — every vector is shifted by the same amount,
  which is what makes the composition a **rotation** rather than something messier. And the
  shift is $2\alpha$, where $\alpha = \theta_2 - \theta_1$ is the angle **between** the
  mirrors.
- The mirrors' own positions have vanished; only their difference survives.

This is the kaleidoscope fact. Two mirrors set $60°$ apart give the familiar six-fold
pattern, not a six-fold *rotation*: the rotation they generate is $120°$, and the six images
are three rotated copies interleaved with three mirrored ones.

Draw it once and watch the doubling.

In [ ]:
def reflect_2d(v: np.ndarray, mirror_angle: float) -> np.ndarray:
    """Reflect the 2-vector v in the LINE through the origin at angle `mirror_angle`."""
    # u is a unit vector along the mirror line. (v @ u) is the length of v's shadow on it,
    # so (v @ u) * u is the part of v ALONG the mirror, which survives, and v minus that is
    # the part ACROSS it, which flips. Keeping the first and negating the second is
    # (v @ u) u - (v - (v @ u) u) = 2 (v @ u) u - v.
    u = np.array([np.cos(mirror_angle), np.sin(mirror_angle)])
    return 2.0 * (v @ u) * u - v


def polar_angle(v: np.ndarray) -> float:
    """The bearing of a 2-vector in degrees, measured anticlockwise from the x-axis."""
    return float(np.rad2deg(np.arctan2(v[1], v[0])))


def arc_points(radius: float, start_deg: float, stop_deg: float) -> tuple:
    """Points along a circular arc, for drawing an angle annotation."""
    t = np.deg2rad(np.linspace(start_deg, stop_deg, 60))
    return radius * np.cos(t), radius * np.sin(t)


mirror_1 = np.deg2rad(15.0)          # first mirror line
mirror_2 = np.deg2rad(55.0)          # second mirror line: alpha = 40 degrees between them
v0 = np.array([np.cos(np.deg2rad(95.0)), np.sin(np.deg2rad(95.0))])
v1 = reflect_2d(v0, mirror_1)        # the image of v in mirror 1
v2 = reflect_2d(v1, mirror_2)        # the image of that image in mirror 2

fig, ax = plt.subplots(figsize=(6.6, 6.0))
for angle, colour, label in ((mirror_1, "#8a8f98", "mirror 1  (15°)"),
                             (mirror_2, "#17797c", "mirror 2  (55°)")):
    d = np.array([np.cos(angle), np.sin(angle)])
    ax.plot([-1.3 * d[0], 1.3 * d[0]], [-1.3 * d[1], 1.3 * d[1]],
            color=colour, lw=1.5, linestyle="--")
    ax.text(1.36 * d[0], 1.36 * d[1], label, color=colour, fontsize=9,
            ha="left", va="center")

for vec, colour, label in ((v0, "#3f6f96", r"$v$"),
                           (v1, "#8fb8d8", r"$v_1 = M_1 v$"),
                           (v2, "#c33b53", r"$v_2 = M_2 v_1$")):
    ax.annotate("", xy=vec, xytext=(0.0, 0.0),
                arrowprops={"arrowstyle": "-|>", "color": colour, "lw": 2.4})
    ax.text(1.16 * vec[0], 1.16 * vec[1], label, color=colour, fontsize=11,
            ha="center", va="center")

ax.plot(*arc_points(0.42, np.rad2deg(mirror_1), np.rad2deg(mirror_2)),
        color="#17797c", lw=1.6)
ax.text(0.56 * np.cos(np.deg2rad(35.0)), 0.56 * np.sin(np.deg2rad(35.0)),
        r"$\alpha = 40°$", color="#17797c", fontsize=12, ha="left", va="center")
ax.plot(*arc_points(0.80, polar_angle(v0), polar_angle(v2)), color="#c33b53", lw=1.6)
ax.text(0.90 * np.cos(np.deg2rad(136.0)), 0.90 * np.sin(np.deg2rad(136.0)),
        r"$2\alpha = 80°$", color="#c33b53", fontsize=12, ha="center", va="bottom")

ax.set_aspect("equal")
ax.set_xlim(-1.55, 1.95)
ax.set_ylim(-1.45, 1.45)
ax.axhline(0.0, color="gray", lw=0.5)
ax.axvline(0.0, color="gray", lw=0.5)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title("Two mirrors 40° apart turn $v$ by 80° — and only the 40° matters")
fig.tight_layout()

print(f"v  is at {polar_angle(v0):7.2f}°")
print(f"v1 is at {polar_angle(v1):7.2f}°   (reflected in mirror 1, the line at 15°)")
print(f"v2 is at {polar_angle(v2):7.2f}°   (then reflected in mirror 2, at 55°)")
print(f"net turn: {polar_angle(v2) - polar_angle(v0):.2f}° = 2 x 40°, and the length is "
      f"unchanged: {np.linalg.norm(v0):.6f} -> {np.linalg.norm(v2):.6f}")

$v$ starts at $95°$ and lands at $175°$: a turn of $80°$, from two mirrors $40°$ apart. The
algebra said $2\alpha$ and the picture agrees.

Three things to take from the figure, because all three survive into 3D and into the
quantum mechanics.

- **The intermediate image is not "half a rotation".** $v_1$ points down and to the right,
  nowhere near either $v$ or $v_2$. Each reflection is a big, handedness-reversing move; the
  rotation is only what *survives* doing two of them.
- **Neither mirror's own position appears in the answer.** Rotate both mirrors by the same
  amount — say both up by $20°$ — and every vector still lands in the same place. Only
  $\alpha$ is real. This is why a rotation can be factored into mirrors in infinitely many
  ways, all of them $\alpha$ apart.
- **Handedness comes back.** One mirror turns a right hand into a left hand; two mirrors
  turn it back. In matrix terms one has determinant $-1$ and the pair has $+1$, which is
  checked explicitly below.

### The same thing in 3D, in plain NumPy

In three dimensions a mirror is a **plane** through the origin, and the cleanest way to name
a plane is by its **unit normal** $\hat n$, the direction sticking out of it. Split any
vector into the part along $\hat n$ and the part lying in the plane; a mirror flips the
first and keeps the second:

$$\text{reflect}(\vec v, \hat n) \;=\; \vec v \;-\; 2(\vec v\cdot\hat n)\,\hat n .$$

Read it as: subtract twice the component along the normal. Subtracting it *once* would land
you in the plane; subtracting it twice carries you out the other side.

Two mirror planes that are not parallel meet in a **line**. Every vector along that line
lies in both planes, so both mirrors leave it alone, so the composition leaves it alone —
that line is the **axis**. And in the plane perpendicular to the axis, the two mirror planes
cut two lines at angle $\alpha$, which is the 2D picture above, giving a turn of $2\alpha$.
That is the whole argument:

$$\text{mirror}_2 \circ \text{mirror}_1 \;=\; \text{rotation by } 2\alpha
  \text{ about } \hat n_1\times\hat n_2, \qquad \cos\alpha = \hat n_1\cdot\hat n_2 .$$

The cross product $\hat n_1\times\hat n_2$ is perpendicular to both normals, so it lies in
both planes: it *is* the intersection line, and its direction fixes the sign of the turn by
the right-hand rule. (Flip which normal you call $\hat n_1$ and the line flips too; the
rotation is unchanged, since a rotation by $2\alpha$ about $\hat u$ equals one by $-2\alpha$
about $-\hat u$.)

There is a theorem behind this — **Cartan–Dieudonné**: every rotation of 3D space is a
product of exactly two reflections, and every orthogonal map of $\mathbb{R}^n$ is a product
of at most $n$ of them. Reflections are the atoms. We will just use the 3D case, and check
it numerically: determinants, the angle from the trace, and the whole matrix against
Rodrigues' formula from Part 1.

In [ ]:
def reflect(v: np.ndarray, n: np.ndarray) -> np.ndarray:
    """Reflect the 3-vector v in the plane through the origin with unit normal n."""
    # (v @ n) is the length of v's shadow on the normal, so (v @ n) * n is that component
    # as a vector. Subtracting it once lands you IN the plane; subtracting it twice carries
    # you through to the far side, which is what a mirror does. Everything perpendicular to
    # n -- that is, everything lying in the mirror plane -- is left completely alone.
    return v - 2.0 * (v @ n) * n


def reflection_matrix(n: np.ndarray) -> np.ndarray:
    """The same reflection as a 3x3 matrix, so we can compose it and take determinants."""
    # np.outer(n, n) is the matrix n n^T, whose action on v is (v . n) n. So I - 2 n n^T
    # applied to v is exactly reflect(v, n), written once and for all v.
    return np.eye(3) - 2.0 * np.outer(n, n)


mirror_rng = np.random.default_rng(1843)
involution_error = 0.0
mirror_det_error = 0.0
trace_angle_error = 0.0
double_reflection_error = 0.0
mirror_slide_error = 0.0

print(f"{'alpha (deg)':>11} {'2*alpha':>8}   {'det M1':>7} {'det M2 M1':>10}   "
      f"{'|M2 M1 - R(n1 x n2, 2a)|':>25}")
for trial in range(200):
    n1 = mirror_rng.normal(size=3)
    n1 /= np.linalg.norm(n1)                   # unit normal of the first mirror plane
    n2 = mirror_rng.normal(size=3)
    n2 /= np.linalg.norm(n2)                   # ...and of the second
    v = mirror_rng.normal(size=3)

    # The angle between two planes is the angle between their normals. np.clip guards the
    # case where a dot product of unit vectors lands at 1 + 1e-16 and arccos returns nan.
    alpha = float(np.arccos(np.clip(n1 @ n2, -1.0, 1.0)))

    m1 = reflection_matrix(n1)
    composed = reflection_matrix(n2) @ m1       # reflect in plane 1 first, then in plane 2

    involution_error = max(involution_error,
                           float(np.abs(reflect(reflect(v, n1), n1) - v).max()))
    mirror_det_error = max(mirror_det_error,
                           abs(float(np.linalg.det(m1)) + 1.0),
                           abs(float(np.linalg.det(composed)) - 1.0))
    # A rotation by phi has trace 1 + 2 cos(phi). The trace is blind to the axis and to the
    # sign of phi, so it is a cheap check of the ANGLE alone -- here, that it is 2*alpha.
    trace_angle_error = max(trace_angle_error,
                            abs(float(np.trace(composed)) - (1.0 + 2.0 * np.cos(2 * alpha))))
    # The full check, which pins the axis and the direction too: Rodrigues' rotation by
    # 2*alpha about the line where the two planes meet, which is n1 x n2.
    predicted = axis_angle_matrix(np.cross(n1, n2), 2.0 * alpha)
    double_reflection_error = max(double_reflection_error,
                                  float(np.abs(composed - predicted).max()))
    # Slide BOTH mirrors around their common line by the same 0.9 rad: same rotation.
    slide = axis_angle_matrix(np.cross(n1, n2), 0.9)
    slid = reflection_matrix(slide @ n2) @ reflection_matrix(slide @ n1)
    mirror_slide_error = max(mirror_slide_error, float(np.abs(slid - composed).max()))

    if trial < 4:
        print(f"{np.rad2deg(alpha):11.3f} {np.rad2deg(2 * alpha):8.3f}   "
              f"{np.linalg.det(m1):7.3f} {np.linalg.det(composed):10.3f}   "
              f"{np.abs(composed - predicted).max():25.2e}")

print(f"\nreflecting twice in one mirror returns v exactly:      {involution_error:.2e}")
print(f"one mirror det -1, two mirrors det +1, worst error:    {mirror_det_error:.2e}")
print(f"the trace says the angle is 2*alpha, worst error:      {trace_angle_error:.2e}")
print(f"the whole matrix is R(n1 x n2, 2*alpha), worst error:  {double_reflection_error:.2e}")
print(f"sliding both mirrors about their line changes nothing: {mirror_slide_error:.2e}")

Everything at $10^{-15}$, over 200 random pairs of planes. Line by line:

- reflecting twice in the *same* mirror is the identity, so a mirror is its own undo;
- one mirror has determinant $-1$ — it reverses handedness, which is why your reflection
  parts its hair on the wrong side — and two mirrors give $+1$, an honest rotation;
- the trace pins the **angle** at $2\alpha$ (a rotation by $\phi$ has trace
  $1 + 2\cos\phi$, which is why the trace alone cannot tell $\phi$ from $-\phi$);
- the full matrix comparison pins the **axis and the direction** too: the composition is
  Rodrigues' rotation by $2\alpha$ about $\hat n_1\times\hat n_2$, to machine precision;
- and sliding both mirrors around their common line by the same amount changes nothing.
  Only the angle between them is real. A rotation does not remember which pair of mirrors
  you factored it into — there are infinitely many, all $\alpha$ apart.

That last point is worth holding onto: **a rotation has no canonical pair of mirrors, but
every pair that works has the same $\alpha = \theta/2$.**

### The quaternion mirror

Here is the step that turns all of this into an explanation of Part 1.

Start from one small fact about Hamilton's product. If $p$ and $q$ are both **pure** —
scalar part zero, so they are really just 3-vectors $\vec p$ and $\vec q$ — then multiplying
them out with $\mathbf{i}^2 = \mathbf{j}^2 = \mathbf{k}^2 = \mathbf{ijk} = -1$ gives

$$p\,q \;=\; -\,(\vec p\cdot\vec q) \;+\; \vec p\times\vec q .$$

Both classical vector products fall out of one quaternion product — which is historically
exactly where the dot and cross products came from; they were carved out of this line.

Now take a **unit** pure quaternion $n$ (so $\lvert\vec n\rvert = 1$, and $n^2 = -1$) and
sandwich a pure $v$ between two copies of it — $n$ on *both* sides, no conjugate:

$$n\,v\,n \;=\; \bigl(-(\vec n\cdot\vec v) + \vec n\times\vec v\bigr)\,n
  \;=\; -(\vec n\cdot\vec v)\,\vec n \;+\; (\vec n\times\vec v)\times\vec n
  \;=\; \vec v - 2(\vec n\cdot\vec v)\,\vec n .$$

(The scalar part cancels, because $(\vec n\times\vec v)\cdot\vec n = 0$: a cross product is
perpendicular to both its factors.) That last expression is `reflect(v, n)`, letter for
letter.

> **A pure unit quaternion is a mirror.** The sandwich $n v n$ reflects $\vec v$ in the plane
> perpendicular to $\vec n$.

Everything now follows in three lines. Reflect in mirror 1, then in mirror 2:

$$v \;\longmapsto\; n_2\,(n_1\,v\,n_1)\,n_2 \;=\; (n_2 n_1)\,v\,(n_1 n_2).$$

Association is all that happened — quaternion multiplication is associative, so the brackets
can be moved. And the right-hand factor is not an accident: conjugation reverses the order of
a product, $\overline{pq} = \bar q\,\bar p$, and a pure quaternion conjugates to *minus*
itself, so

$$\overline{n_2 n_1} \;=\; \bar n_1\,\bar n_2 \;=\; (-n_1)(-n_2) \;=\; n_1 n_2 .$$

The two minus signs cancel. So the double reflection is exactly the rotation sandwich
$q\,v\,\bar q$ of Part 1, with

$$\boxed{\;q \;=\; n_2\,n_1\;}$$

**The quaternion of a rotation is the Hamilton product of its two mirror normals.** And its
four numbers are now readable: by the pure-product rule,

$$q \;=\; n_2 n_1 \;=\; -(\vec n_1\cdot\vec n_2) \;+\; \vec n_2\times\vec n_1
   \;=\; -\cos\alpha \;+\; \vec n_2\times\vec n_1 ,$$

whose vector part has length $\sin\alpha$ and points along $-(\vec n_1\times\vec n_2)$ — the
intersection line. Compare with Part 1's $\cos\tfrac{\theta}{2} + \sin\tfrac{\theta}{2}\hat n$
at $\theta = 2\alpha$: the two agree **up to an overall minus sign**, which is precisely the
sign the sandwich cannot see. Check all of it.

In [ ]:
def pure_quaternion(v: np.ndarray) -> np.ndarray:
    """Embed a 3-vector as a pure quaternion: scalar part 0, vector part v."""
    return np.array([0.0, v[0], v[1], v[2]])


quat_mirror_rng = np.random.default_rng(1927)
pure_sandwich_error = 0.0
mirror_quaternion_error = 0.0
mirror_parts_error = 0.0
sign_freedom_error = 0.0

for _ in range(200):
    n1 = quat_mirror_rng.normal(size=3)
    n1 /= np.linalg.norm(n1)
    n2 = quat_mirror_rng.normal(size=3)
    n2 /= np.linalg.norm(n2)
    v = quat_mirror_rng.normal(size=3)
    alpha = float(np.arccos(np.clip(n1 @ n2, -1.0, 1.0)))
    composed = reflection_matrix(n2) @ reflection_matrix(n1)

    # (i) ONE mirror. The sandwich n v n -- note: n on both sides, no conjugate -- comes
    #     back pure (scalar part 0) and equals reflect(v, n).
    one_mirror = hamilton(hamilton(pure_quaternion(n1), pure_quaternion(v)),
                          pure_quaternion(n1))
    pure_sandwich_error = max(pure_sandwich_error,
                              abs(float(one_mirror[0])),
                              float(np.abs(one_mirror[1:] - reflect(v, n1)).max()))

    # (ii) TWO mirrors nest into the rotation sandwich, with q = n2 * n1.
    q = hamilton(pure_quaternion(n2), pure_quaternion(n1))
    mirror_quaternion_error = max(
        mirror_quaternion_error,
        float(np.abs(rotate_by_quaternion(q, v) - composed @ v).max()))

    # (iii) Its four numbers are exactly the two things a pair of normals can make:
    #       scalar part -(n1 . n2) = -cos(alpha), vector part n2 x n1 of length sin(alpha).
    mirror_parts_error = max(mirror_parts_error,
                             abs(float(q[0]) + np.cos(alpha)),
                             float(np.abs(q[1:] - np.cross(n2, n1)).max()),
                             abs(float(np.linalg.norm(q[1:])) - np.sin(alpha)))

    # (iv) ...which is Part 1's axis-angle quaternion for a turn of 2*alpha about the
    #      intersection line n1 x n2 -- up to the overall sign, hence the + not a -.
    mirror_quaternion_error = max(
        mirror_quaternion_error,
        float(np.abs(q + axis_angle_quaternion(np.cross(n1, n2), 2.0 * alpha)).max()))

    # (v) The sign freedom, made concrete: call the OTHER side of mirror 1 the front.
    #     Same plane, same reflection, same composed rotation -- and q flips sign.
    flipped = hamilton(pure_quaternion(n2), pure_quaternion(-n1))
    sign_freedom_error = max(
        sign_freedom_error,
        float(np.abs(reflection_matrix(-n1) - reflection_matrix(n1)).max()),
        float(np.abs(flipped + q).max()))

print(f"n v n reproduces reflect(v, n), worst over 200 pairs:     {pure_sandwich_error:.2e}")
print(f"q = n2 n1 rotates exactly as the two mirrors do:          {mirror_quaternion_error:.2e}")
print(f"  its parts are -cos(alpha) and n2 x n1:                  {mirror_parts_error:.2e}")
print(f"flipping one normal: same mirrors, same rotation, -q:     {sign_freedom_error:.2e}")

# One pair spelled out: two mirrors 22.5 degrees apart, both normals in the xy-plane, so
# the planes meet along z and the rotation must be 45 degrees about z.
n1 = np.array([1.0, 0.0, 0.0])
n2 = np.array([np.cos(np.pi / 8), np.sin(np.pi / 8), 0.0])
print("\nmirrors 22.5 deg apart, normals in the xy-plane:")
print("  q = n2 * n1                 =",
      np.array2string(hamilton(pure_quaternion(n2), pure_quaternion(n1)), precision=6))
print("  quaternion of 45 deg about z =",
      np.array2string(axis_angle_quaternion(np.array([0.0, 0.0, 1.0]), np.pi / 4),
                      precision=6))
print("  ...the same rotation, opposite overall sign.")

### The half-angle, finally

Put the two facts side by side.

1. Two mirrors whose planes meet at angle $\alpha$ compose to a rotation by $2\alpha$.
2. The quaternion of that rotation is $q = n_2 n_1$ — the Hamilton product of the two
   mirror normals — whose scalar part is $\pm\cos\alpha$ and whose vector part has length
   $\sin\alpha$.

Now read the standard formula again:

$$q \;=\; \cos\tfrac{\theta}{2} \;+\; \sin\tfrac{\theta}{2}\,\hat n .$$

**$\theta/2$ is the angle between the mirrors.** Not a parametrisation quirk, not a
convention someone picked to make a formula come out neat — a measured angle in the figure
above. To turn the world by $\theta$ you need two mirrors $\theta/2$ apart, and the
quaternion *is* those two mirrors, multiplied. The half is forced by counting: the rotation
is the product of two things, so each of them carries half the job.

Everything else Part 1 asked you to swallow is forced along with it:

- **Why a sandwich, and not a product?** Because there are two mirrors, and a mirror is
  itself a sandwich — $n v n$ — since reflecting must leave the plane alone while flipping
  the normal, and one-sided multiplication cannot do that.
- **Why $\bar q$ on the right?** Because $\overline{n_2 n_1} = \bar n_1 \bar n_2 = n_1 n_2$:
  conjugation reverses order, and the nested mirrors deliver the two normals in reversed
  order on the right. The bar is not a decoration; it is the reversal.
- **Why $q$ and $-q$?** Because each mirror has two unit normals and nothing distinguishes
  them. Choosing which side of a plane is "front" — twice, independently — is a choice of
  two signs, and their product is the one sign the rotation cannot see.

That last bullet is the double cover, and it will come back as physics in Part 5.

### Coda — the closest thing to a mirror a circuit can do

Everything above was 1843 geometry, and the Bloch sphere is a sphere in that same 3D space.
So it is fair to ask: which gates are the mirrors?

None of them. Every gate acts on the Bloch sphere as a *rotation* — determinant $+1$,
handedness preserved — and a mirror has determinant $-1$. But the next best thing exists,
and you have been using it since your first circuit: a gate that is both **Hermitian**
($U = U^\dagger$, equal to its own conjugate transpose) and **unitary** ($U U^\dagger =
\mathbb{1}$). Those two conditions together say $U^2 = \mathbb{1}$: the gate is its own undo.
`X`, `Y`, `Z` and `H` are all of this kind.

Every such gate except the identity can be written, up to an overall phase, as
$\hat n\cdot\vec\sigma = n_x\sigma_x + n_y\sigma_y + n_z\sigma_z$ for some unit vector
$\hat n$, where $\sigma_x, \sigma_y, \sigma_z$ are the **Pauli matrices** — the three
$2\times2$ matrices whose expectation values are exactly the three coordinates of the Bloch
vector that `inspect.bloch_vector` returns:

$$\sigma_x = \begin{pmatrix}0&1\\1&0\end{pmatrix},\quad
\sigma_y = \begin{pmatrix}0&-i\\i&0\end{pmatrix},\quad
\sigma_z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}.$$

So `X` is $\hat x\cdot\vec\sigma$, `Z` is $\hat z\cdot\vec\sigma$, and `H` is
$\tfrac{1}{\sqrt2}(\sigma_x + \sigma_z)$ — the unit vector halfway between $\hat x$ and
$\hat z$. (Part 3 will show that these three matrices *are* $\mathbf{i}, \mathbf{j},
\mathbf{k}$ wearing a factor of $-i$; here we only need what they do to the sphere.)

Since such a gate is its own inverse, applying it to a state conjugates that state's Pauli
decomposition by it, and the algebra gives

$$(\hat n\cdot\vec\sigma)\,(\vec v\cdot\vec\sigma)\,(\hat n\cdot\vec\sigma)
  \;=\; \bigl(2(\hat n\cdot\vec v)\,\hat n - \vec v\bigr)\cdot\vec\sigma .$$

Look at the bracket: it is $-\bigl(\vec v - 2(\hat n\cdot\vec v)\hat n\bigr)$, which is
**minus the mirror**. The axis $\hat n$ survives and everything perpendicular to it flips —
a $\pi$-rotation about $\hat n$. That is a mirror composed with the central inversion
$\vec v \mapsto -\vec v$, and since inversion also has determinant $-1$, the two minus signs
multiply to $+1$ and the result is an honest rotation. As close to a mirror as a gate gets:
the reflection is there, with an invisible extra minus sign bolted on.

Check it on all four, against the `reflect` we already wrote.

In [ ]:
def bloch_after(op) -> np.ndarray:
    """Prepare one fixed, generic qubit state, apply `op` to it, return its Bloch vector.

    The preparation is the same every call, so the only thing that differs between rows
    below is the gate. The two rotations put the starting vector off all three axes, where
    a wrong prediction has nowhere to hide.
    """
    qc_mirror = Circuit(name="mirror", seed=0)
    w = qc_mirror.alloc("q")
    Ry(w, theta=0.7)
    Rz(w, theta=1.3)
    op(w)
    return np.array(qc_mirror.inspect.bloch_vector(w))


start_v = bloch_after(lambda _w: None)          # no gate at all: the starting orientation
hermitian_unitaries = {
    "X": (X, np.array([1.0, 0.0, 0.0])),
    "Y": (Y, np.array([0.0, 1.0, 0.0])),
    "Z": (Z, np.array([0.0, 0.0, 1.0])),
    "H": (H, np.array([1.0, 0.0, 1.0]) / np.sqrt(2.0)),   # halfway between x and z
}

print("starting Bloch vector:", np.array2string(start_v, precision=4))
print(f"\n{'gate':>4} {'axis n':>22}   {'Bloch vector after':>26}   {'2(n.v)n - v':>26}")
mirror_gate_error = 0.0
for name, (gate_fn, n) in hermitian_unitaries.items():
    after = bloch_after(gate_fn)
    # -reflect(v, n) = 2 (n.v) n - v: the mirror, then the central inversion v -> -v.
    minus_mirror = -reflect(start_v, n)
    mirror_gate_error = max(mirror_gate_error, float(np.abs(after - minus_mirror).max()))
    print(f"{name:>4} {np.array2string(n, precision=3):>22}   "
          f"{np.array2string(after, precision=4):>26}   "
          f"{np.array2string(minus_mirror, precision=4):>26}")

print(f"\nworst disagreement: {mirror_gate_error:.2e}")


def h_x_h(w) -> None:
    """H X H, spelled with qsim's conjugation combinator: do H, then X, then undo H."""
    with qsim.within(H, w):
        X(w)


hxh_error = float(np.abs(bloch_after(h_x_h) - bloch_after(Z)).max())
print(f"H X H and Z move the Bloch vector identically, to {hxh_error:.2e}")

Four rows, four exact matches: each of these gates does "keep the axis, flip everything
perpendicular to it" to the Bloch vector.

Read the $H$ row geometrically. $H$'s axis is the unit vector halfway between $\hat x$ and
$\hat z$, so conjugating by $H$ exchanges the $x$ and $z$ axes — which is exactly what
$HXH = Z$ says in gate language, and what "to measure along $x$, rotate $x$ onto $z$ and
measure along $z$" says in laboratory language. **You have been using mirrors all along.**

Now compose two of them. Keep $Z$ ($\pi$ about $\hat z$) as the first, and build the second
by *tilting* $Z$: conjugating a gate by $R_y(\varphi)$ carries its axis through $\varphi$ in
the $xz$-plane, and qsim spells conjugation with `within`:

```python
with qsim.within(Ry, q, theta=phi):
    Z(q)
```

Two $\pi$-gates whose axes sit $\varphi$ apart, both axes in the $xz$-plane — so their
"mirror planes" meet along $\hat y$, and the two-mirror rule predicts a rotation about
$\hat y$ by $2\varphi$. Sweep $\varphi$ and read the turn off the Bloch vector.

In [ ]:
def tilted_pi_gate(w, phi: float) -> None:
    """A pi-rotation about the axis you get by tilting z through phi in the xz-plane."""
    # `within(V, ...)` does V, then the body, then V undone -- so this is the conjugation
    # Ry(phi)^-1 Z Ry(phi), which is a pi-rotation about the tilted axis, not about z.
    with qsim.within(Ry, w, theta=phi):
        Z(w)


phis = np.linspace(0.0, np.pi / 2.0, 13)
two_phi_error = 0.0

print(f"{'phi':>7} {'2*phi':>7}   {'Bloch vector after both pi-gates':>34}   {'turn':>8}")
for phi in phis:
    qc_mirrors = Circuit(name="two-mirrors", seed=0)
    w = qc_mirrors.alloc("q")
    Z(w)                                    # mirror-gate 1: pi about z, from |0>
    tilted_pi_gate(w, float(phi))           # mirror-gate 2: pi about the tilted axis
    v = np.array(qc_mirrors.inspect.bloch_vector(w))
    # |0> sits at the north pole (0, 0, 1), which is perpendicular to y, and the composition
    # turns about y -- so the motion stays in the xz-plane and the turn is just the polar
    # angle of (x, z). arctan2(x, z) measures that angle from +z, signed.
    turn = float(np.arctan2(v[0], v[2]))
    two_phi_error = max(two_phi_error, abs(abs(turn) - 2.0 * float(phi)))
    if int(round(phi / phis[1])) % 3 == 0:
        print(f"{phi:7.4f} {2 * phi:7.4f}   {np.array2string(v, precision=5):>34}   "
              f"{turn:+8.4f}")

print(f"\nworst | |turn| - 2*phi | over the sweep: {two_phi_error:.2e}")

Exactly $2\varphi$, at every point of the sweep, to $4\times10^{-16}$.

The sign is worth one line, because it is the same rule as before. `within(Ry, q, theta=φ)`
runs $R_y(\varphi)$, then the body, then $R_y(\varphi)^{-1}$, so the tilted gate's axis is
$\hat z$ carried along by $R_y(-\varphi)$: it leans toward $-x$. The two axes are therefore
$\hat n_1 = \hat z$ and $\hat n_2 = (-\sin\varphi,\,0,\,\cos\varphi)$, whose cross product
$\hat n_1 \times \hat n_2$ points along $-\hat y$ — and the printed turn is $-2\varphi$,
swinging toward $-x$. The mirror rule fixed the direction, not just the size.

Two closing notes, one sentence each.

- **The $2\varphi$ law is the two-mirrors law again, living entirely inside the rotations:**
  each $\pi$-gate is a mirror composed with the central inversion, and composing two of them
  cancels the two inversions, leaving exactly the pair of mirrors whose product this part
  showed to be a rotation by twice the angle between them.
- **A true reflection of the Bloch sphere — determinant $-1$ — is not any gate at all:** it
  needs an **anti-unitary** operation, complex conjugation of the state, which is how quantum
  mechanics represents *time reversal*; that is a door we point at here and do not open.

So the half-angle, the sandwich, the $q$-versus-$-q$ ambiguity and the $\pi$-gates you use
every day are one fact in four costumes: **rotations are made of mirrors, and a quaternion is
the pair.** With that settled, we can go back to qubits and check the dictionary in detail.

## Part 3 — The dictionary

Now the claim. A one-qubit gate is a $2\times2$ complex matrix — eight real numbers.
Requiring it to be **unitary** (length-preserving, so that total probability stays 1)
imposes four real conditions and leaves four. Requiring its determinant to be exactly $1$ —
a harmless normalisation that picks one representative out of each family of matrices
differing by an overall phase — costs one more, leaving three.

So one-qubit gates form a three-parameter family, with a composition law that does not
commute. And the most natural way to coordinatise it turns out to be four real numbers
subject to $a^2 + b^2 + c^2 + d^2 = 1$. We have met that object already: it was Part 1.

The dictionary is:

$$\boxed{\;U \;=\; a\,\mathbb{1} \;-\; i\bigl(b\,\sigma_x + c\,\sigma_y + d\,\sigma_z\bigr)
\qquad\longleftrightarrow\qquad q \;=\; a + b\mathbf{i} + c\mathbf{j} + d\mathbf{k}\;}$$

where $\sigma_x, \sigma_y, \sigma_z$ are the **Pauli matrices** met in Part 2's coda — the
three $2\times2$ matrices that measure the three Bloch-sphere axes, and whose expectation
values *are* the Bloch vector $(x, y, z)$ that `inspect.bloch_vector` returns:

$$\sigma_x = \begin{pmatrix}0&1\\1&0\end{pmatrix},\quad
\sigma_y = \begin{pmatrix}0&-i\\i&0\end{pmatrix},\quad
\sigma_z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}.$$

Note the $-i$ out front. That single factor is what turns three matrices that *square to
$+1$* into three objects that *square to $-1$* — which is the defining property of
$\mathbf{i}, \mathbf{j}, \mathbf{k}$. Check it: $(-i\sigma_x)^2 = -\sigma_x^2 = -\mathbb{1}$.
And $(-i\sigma_x)(-i\sigma_y) = -\sigma_x\sigma_y = -i\sigma_z$, which is the $+\mathbf{k}$
of $\mathbf{ij} = \mathbf{k}$. The Pauli matrices are Hamilton's units in disguise, and the
disguise is one factor of $-i$.

Multiplying the dictionary out gives

$$U = \begin{pmatrix} a - i d & -c - i b \\ c - i b & a + i d\end{pmatrix},$$

so going the other way needs no linear algebra at all: the four real numbers are sitting in
the top row.

### Getting the matrices out of qsim

One wrinkle. qsim deliberately does not expose its gate matrices — the library's central
rule is that it never builds a matrix bigger than one gate, and a public "give me your
matrix" accessor would invite exactly the habit it is avoiding. So we recover them the
honest way, the way an experimentalist would: **feed the gate each basis state and record
what comes out.** $U\lvert 0\rangle$ *is* the first column of $U$; $U\lvert 1\rangle$ is the
second. Two circuit runs per matrix.

In [ ]:
def gate_matrix(apply_gate) -> np.ndarray:
    """The 2x2 matrix of a one-qubit gate, read out of qsim one column at a time.

    `apply_gate` is a function taking a qubit handle, e.g. `lambda q: Rx(q, theta=0.4)`.
    """
    columns = []
    for basis_bit in (0, 1):
        qc = Circuit(name="probe", seed=0)
        q = qc.alloc("q")
        if basis_bit:
            X(q)              # prepare |1> instead of |0>
        apply_gate(q)
        columns.append(qc.inspect.state_vector())
    # U|0> and U|1> are the first and second columns of U, so stacking the two output
    # state vectors side by side (axis=1 makes them columns, not rows) rebuilds U.
    return np.stack(columns, axis=1)


PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
PAULI_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)


def from_quaternion(q: np.ndarray) -> np.ndarray:
    """The 2x2 unitary matrix for the quaternion q = a + bi + cj + dk."""
    a, b, c, d = q
    return a * np.eye(2, dtype=complex) - 1j * (b * PAULI_X + c * PAULI_Y + d * PAULI_Z)


def to_quaternion(u: np.ndarray) -> np.ndarray:
    """The quaternion [a, b, c, d] of a 2x2 unitary, read straight off its entries.

    Since  U = [[a - i d,  -c - i b],
                [c - i b,   a + i d]],
    the top row alone determines all four numbers.
    """
    return np.array([u[0, 0].real, -u[0, 1].imag, -u[0, 1].real, -u[0, 0].imag])


# Sanity check that the two directions are inverse to each other.
sample_q = axis_angle_quaternion(np.array([1.0, 2.0, -3.0]), 0.9)
print("round trip q -> U -> q reproduces q:",
      np.allclose(to_quaternion(from_quaternion(sample_q)), sample_q))

### Prediction 1 — every rotation gate is a unit quaternion, with the axis you expect

If the dictionary is right, then `Rx(theta)` — whose whole job is "rotate the Bloch vector
by $\theta$ about $x$" — must map to *precisely* the axis–angle quaternion of Part 1:

$$R_x(\theta) \;\longleftrightarrow\; \cos\tfrac{\theta}{2} + \sin\tfrac{\theta}{2}\,\mathbf{i},$$

and likewise $R_y \to \mathbf{j}$, $R_z \to \mathbf{k}$. Note that we are *not* going to
hand the quaternion any $\theta/2$: we hand it $\theta$, ask the same rotation of both, and
see whether the half-angle appears on its own.

In [ ]:
gate_axes = {
    "Rx": (Rx, np.array([1.0, 0.0, 0.0])),
    "Ry": (Ry, np.array([0.0, 1.0, 0.0])),
    "Rz": (Rz, np.array([0.0, 0.0, 1.0])),
}
angles = [0.0, np.pi / 4, np.pi / 2, np.pi, 3.0 * np.pi / 2, 2.0 * np.pi]

component_error = 0.0
norm_error = 0.0

print(f"{'gate':>5} {'theta':>7}   {'a':>8} {'b':>8} {'c':>8} {'d':>8}   {'||q||':>6}"
      f"   {'cos(t/2)':>9} {'sin(t/2)':>9}")
for name, (gate, axis) in gate_axes.items():
    for theta in angles:
        u = gate_matrix(lambda q, g=gate, t=theta: g(q, theta=t))
        q = to_quaternion(u)
        predicted = axis_angle_quaternion(axis, theta)
        component_error = max(component_error, float(np.abs(q - predicted).max()))
        norm_error = max(norm_error, abs(quat_norm(q) - 1.0))
        print(f"{name:>5} {theta:7.3f}   {q[0]:8.4f} {q[1]:8.4f} {q[2]:8.4f} {q[3]:8.4f}"
              f"   {quat_norm(q):6.4f}   {np.cos(theta / 2):9.4f} {np.sin(theta / 2):9.4f}")

print(f"\nlargest deviation from (cos(theta/2), sin(theta/2) * axis): {component_error:.2e}")
print(f"largest deviation of ||q|| from 1:                          {norm_error:.2e}")

Read the last two columns against the first four. `Rx` puts $\cos(\theta/2)$ in $a$ and
$\sin(\theta/2)$ in $b$ — the $\mathbf{i}$ slot — and nothing anywhere else. `Ry` fills
$\mathbf{j}$, `Rz` fills $\mathbf{k}$. Every one has norm exactly 1.

**This is the answer to "why does the Bloch sphere use half-angles?"** It is not a
convention someone chose, and it is not a quantum peculiarity. Rotations of 3D space are
*already* parametrised by half-angles the moment you insist on representing them by
something that multiplies associatively and never degenerates — which is what Hamilton
found in 1843 and what the Pauli matrices rediscovered in 1927. The half-angle was in the
geometry the whole time. Quantum mechanics just handed us a physical system whose state
lives in the four-number layer rather than the three-number one.

Look at the last row of each block, $\theta = 2\pi$: the quaternion is $(-1, 0, 0, 0)$. That
is $-1$, not $1$. A full turn.

### Prediction 2 — matrix product $=$ Hamilton product

A dictionary between two sets of objects is only interesting if it also translates the
*verbs*. The verb here is composition: doing one rotation after another. On the matrix side
that is $U_1 U_2$; on the quaternion side it is `hamilton(q1, q2)`. The claim is that they
are the same operation, with the same ordering convention (rightmost acts first).

If this holds, the map is a **group isomorphism**: unit quaternions and the group $SU(2)$ of
one-qubit gates are literally the same group wearing different notation.

The test: 300 random pairs of qsim rotation gates, with angles drawn from $[0, 4\pi)$ so
that the double-cover region gets exercised too.

In [ ]:
pair_rng = np.random.default_rng(2024)
gate_list = [Rx, Ry, Rz]
product_error = 0.0

for _ in range(300):
    g1 = gate_list[pair_rng.integers(3)]
    g2 = gate_list[pair_rng.integers(3)]
    t1 = float(pair_rng.uniform(0.0, 4.0 * np.pi))
    t2 = float(pair_rng.uniform(0.0, 4.0 * np.pi))
    u1 = gate_matrix(lambda q, g=g1, t=t1: g(q, theta=t))
    u2 = gate_matrix(lambda q, g=g2, t=t2: g(q, theta=t))
    # Translate-then-multiply versus multiply-then-translate. If the two agree for every
    # pair, the dictionary is a homomorphism, not just a coincidence of components.
    via_matrices = to_quaternion(u1 @ u2)
    via_quaternions = hamilton(to_quaternion(u1), to_quaternion(u2))
    product_error = max(product_error, float(np.abs(via_matrices - via_quaternions).max()))

print(f"largest disagreement over 300 random pairs: {product_error:.3e}")

# One concrete example, printed in full, so the claim is not just a max-error number.
ux = gate_matrix(lambda q: Rx(q, theta=0.7))
uy = gate_matrix(lambda q: Ry(q, theta=1.9))
print("\nRx(0.7) then Ry(1.9)")
print("  quaternion of the product matrix :",
      np.array2string(to_quaternion(uy @ ux), precision=6))
print("  Hamilton product of the two      :",
      np.array2string(hamilton(to_quaternion(uy), to_quaternion(ux)), precision=6))
print("  ...and in the other order        :",
      np.array2string(hamilton(to_quaternion(ux), to_quaternion(uy)), precision=6))

Agreement to $10^{-15}$ across 300 random pairs, and the third line shows the two orders
genuinely differ — the non-commutativity survives the translation, as it must.

So: **the group of one-qubit gates and the group of unit quaternions are the same group.**
Not similar. The same. Every fact you know about one is a fact about the other.

## Part 4 — Both of them rotate the same vector

An isomorphism of groups is an algebraic statement. Here is the geometric one, which is
what you would actually feel if you could hold a qubit.

Prepare a qubit in some arbitrary state with a seeded pile of `Ry`/`Rz` rotations, and read
its Bloch vector once with `inspect.bloch_vector`. From that moment we run two experiments
side by side:

- **qsim**, applying real rotation gates to the real quantum state and reporting the Bloch
  vector after each;
- **NumPy**, taking that one recorded 3-vector and rotating it by quaternion sandwiches —
  never looking at the quantum state again.

The NumPy track never resynchronises. If the two agree after six chained rotations, they
agree because the quaternion sandwich *is* what a rotation gate does to a Bloch vector.

In [ ]:
qc = Circuit(name="spin", seed=11)
q = qc.alloc("q")

prep_rng = np.random.default_rng(7)
for _ in range(3):                              # an arbitrary starting orientation
    Ry(q, theta=float(prep_rng.uniform(0.0, 2.0 * np.pi)))
    Rz(q, theta=float(prep_rng.uniform(0.0, 2.0 * np.pi)))

predicted_v = np.array(qc.inspect.bloch_vector(q))   # the one and only reading we borrow
print("starting Bloch vector:", np.array2string(predicted_v, precision=4))
print()

steps = []
bloch_error = 0.0
print(f"{'step':>4} {'gate':>4} {'theta':>7}   {'qsim (x, y, z)':>28}   "
      f"{'quaternion (x, y, z)':>28}")
for step in range(6):
    name = ["Rx", "Ry", "Rz"][step % 3]
    gate, axis = gate_axes[name]
    theta = float(prep_rng.uniform(0.0, 2.0 * np.pi))

    gate(q, theta=theta)                                     # the quantum track
    measured_v = np.array(qc.inspect.bloch_vector(q))

    u = gate_matrix(lambda w, g=gate, t=theta: g(w, theta=t))
    predicted_v = rotate_by_quaternion(to_quaternion(u), predicted_v)   # the numpy track

    bloch_error = max(bloch_error, float(np.abs(measured_v - predicted_v).max()))
    steps.append((name, theta, measured_v, predicted_v))
    print(f"{step:>4} {name:>4} {theta:7.3f}   "
          f"{np.array2string(measured_v, precision=4):>28}   "
          f"{np.array2string(predicted_v, precision=4):>28}")

print(f"\nlargest disagreement after six chained rotations: {bloch_error:.2e}")

The two right-hand columns are the same numbers to every digit printed. The same picture,
with the eighteen numbers side by side: bar height is what qsim's quantum state says, the
cross is what Hamilton's arithmetic predicted.

In [ ]:
# One bar per (step, component): the bar height is qsim's Bloch component, the cross is
# what the quaternion sandwich predicted. Eighteen pairs, no visible daylight between them.
positions = np.arange(18)
qsim_values = np.concatenate([measured for _, _, measured, _ in steps])
quat_values = np.concatenate([predicted for _, _, _, predicted in steps])

fig, ax = plt.subplots(figsize=(10.0, 3.6))
ax.bar(positions, qsim_values, width=0.62, color="#8fb8d8",
       edgecolor="#3f6f96", label="qsim: inspect.bloch_vector")
ax.plot(positions, quat_values, "kx", markersize=8, markeredgewidth=1.6,
        linestyle="none", label=r"numpy: $q\,v\,\bar q$")
ax.axhline(0.0, color="gray", lw=0.6)
for boundary in range(1, 6):
    ax.axvline(3 * boundary - 0.5, color="gray", lw=0.5, linestyle=":")
ax.set_xticks(positions, [c for _ in steps for c in "xyz"])
ax.set_xlabel("Bloch component, grouped by rotation step (0 … 5)")
ax.set_ylim(-1.35, 1.45)
ax.legend(fontsize=9, ncol=2, loc="upper center")
ax.set_title("Six chained rotations: the quantum state and a bare 3-vector stay in step")
fig.tight_layout()

Every cross lands on top of its bar, to $10^{-15}$ or so. The NumPy track has not seen a
quantum state since the first line; it has been pushing three real numbers around with
Hamilton's 1843 arithmetic, and it tracks the qubit exactly.

This is worth stating carefully, because it is the point where a beginner usually stops
being confused about the Bloch sphere. The Bloch sphere is *not* a picture invented to make
qubits look friendly. A qubit's state space really is the unit quaternions, and the Bloch
sphere is what you see when you look at that space through the sandwich $q v \bar q$ — the
map that forgets exactly one thing. What it forgets is the subject of the next section.

## Part 5 — The double cover: what the sandwich forgets

We noted in Part 1 that $q$ and $-q$ produce the same rotation, because the sandwich uses
$q$ twice, and Part 2 said what the two signs *are*: which side of each mirror you decided
to call the front. Run that observation as an experiment.

Sweep $\theta$ from $0$ to $4\pi$ — two full turns — for $R_x$, and plot two things on one
figure:

- the **Bloch vector** the qubit actually has, which is what any measurement could tell you;
- the **quaternion components** $a = \cos(\theta/2)$ and $b = \sin(\theta/2)$, which is what
  the state vector is doing underneath.

In [ ]:
sweep = np.linspace(0.0, 4.0 * np.pi, 241)
bloch_rows = []
quat_rows = []

for theta in sweep:
    qc_sweep = Circuit(name="cover", seed=0)
    qq = qc_sweep.alloc("q")
    Rx(qq, theta=float(theta))
    bloch_rows.append(qc_sweep.inspect.bloch_vector(qq))
    quat_rows.append(to_quaternion(gate_matrix(lambda w, t=theta: Rx(w, theta=float(t)))))

# Lists of tuples become (241, 3) and (241, 4) arrays; column k is one component
# as a function of theta.
bloch_sweep = np.array(bloch_rows)
quat_sweep = np.array(quat_rows)

fig, ax = plt.subplots(figsize=(9.5, 4.0))
ax.plot(sweep, bloch_sweep[:, 2], lw=2.4, color="#3f6f96", label="Bloch z  (period 2π)")
ax.plot(sweep, bloch_sweep[:, 1], lw=2.4, color="#8fb8d8", label="Bloch y  (period 2π)")
ax.plot(sweep, quat_sweep[:, 0], "k--", lw=1.6, label=r"quaternion $a=\cos(\theta/2)$")
ax.plot(sweep, quat_sweep[:, 1], ":", color="#c33b53", lw=1.8,
        label=r"quaternion $b=\sin(\theta/2)$")
ax.axvline(2.0 * np.pi, color="#c33b53", lw=1.0)
ax.axhline(0.0, color="gray", lw=0.6)
ax.annotate("one full turn: Bloch home, quaternion at −1",
            xy=(2.0 * np.pi, -1.02), xytext=(2.0 * np.pi + 0.35, -1.5), fontsize=9,
            ha="left", va="center", color="#c33b53",
            arrowprops={"arrowstyle": "->", "color": "#c33b53", "lw": 1.0})
ax.set_xticks([0, np.pi, 2 * np.pi, 3 * np.pi, 4 * np.pi], ["0", "π", "2π", "3π", "4π"])
ax.set_xlabel(r"$\theta$ in $R_x(\theta)$")
ax.set_ylim(-1.8, 1.65)
ax.legend(fontsize=9, ncol=2, loc="upper center")
ax.set_title(r"$SU(2) \to SO(3)$ is two-to-one: the sphere comes home twice as often")
fig.tight_layout()

print("at theta = 2π  the quaternion is", np.array2string(quat_sweep[120], precision=6))
print("at theta = 4π  the quaternion is", np.array2string(quat_sweep[240], precision=6))

The two solid curves — everything observable — complete a cycle at $2\pi$ and do it again.
The two dashed curves do not: at $\theta = 2\pi$ the quaternion sits at $(-1, 0, 0, 0)$,
which is $-1$, and only at $4\pi$ does it return to $+1$.

The name for this is the **double cover**. The map from unit quaternions (equivalently
$SU(2)$, equivalently one-qubit gates) to ordinary 3D rotations ($SO(3)$) is two-to-one:
$q$ and $-q$ send down to the same rotation. Going once around a loop of rotations lifts to
a path that ends at $-q$; going around twice brings you back.

Two ways to say the same thing:

- **Geometrically.** The space of 3D rotations is not simply connected — there is a loop in
  it that cannot be shrunk to a point, and a rotation by $2\pi$ traverses it. Quaternion
  space is the simply-connected space sitting above it, wrapped twice.
- **Physically.** $R_x(2\pi) = -\mathbb{1}$, exactly as the closing cell of
  [one_qubit_playground](one_qubit_playground.ipynb) found. **$q$ and $-q$ are the same
  rotation but they are not the same quantum operation.**

That second sentence is the crux, and it is where most treatments stop, with a note that the
$-1$ is "just a global phase, unobservable". The next section takes that seriously and then
breaks it.

## Part 6 — Cashing out the minus sign

First, the honest part. On a single qubit, all by itself, the $-1$ really is invisible.

Every probability is $|{\rm amplitude}|^2$, and $|-z|^2 = |z|^2$. Multiply the whole state
by $-1$ and every measurement statistic on every axis is untouched. Below, two circuits — one
that has been turned through $2\pi$ and one that has not — sampled from the same seed.

In [ ]:
def turned_qubit(theta: float) -> Circuit:
    """One qubit, rotated about x by theta. Same seed every time."""
    qc_one = Circuit(name="phase", seed=5)
    qubit = qc_one.alloc("q")
    Rx(qubit, theta=theta)
    return qc_one


turned = turned_qubit(2.0 * np.pi)   # a full 360-degree turn
still = turned_qubit(0.0)            # nothing at all

print("after a 2π turn :", turned.inspect.ket())
print("after nothing   :", still.inspect.ket())
print()
print("Bloch vectors identical:",
      np.allclose(turned.inspect.bloch_vector(turned.qubits[0]),
                  still.inspect.bloch_vector(still.qubits[0])))
turned_counts = turned.inspect.sample(2000)
still_counts = still.inspect.sample(2000)
print("2000 samples each, tallies literally equal:", turned_counts == still_counts)
print("  turned:", dict(turned_counts), " still:", dict(still_counts))

The state vectors are visibly different — $-1.000\lvert 0\rangle$ against
$1.000\lvert 0\rangle$ — and no experiment on this qubit can tell them apart. So far the
textbook is right.

### Now make it a *relative* phase

The trick is the one every quantum algorithm runs on, and the one
[one_qubit_playground](one_qubit_playground.ipynb) built its interferometer from: a phase
that is invisible on the whole state becomes an *outcome* the moment it applies to only part
of a superposition.

So we rotate the qubit through $2\pi$ **conditionally**. Put a control qubit $c$ into
$\lvert +\rangle$ with a Hadamard, and run the $2\pi$ turn only on the branch where
$c = 1$:

```python
H(c)
with qc.control(c):
    Rx(t, theta=2*np.pi)
H(c)
```

Follow the amplitudes. After the first Hadamard the two qubits are in

$$\tfrac{1}{\sqrt2}\bigl(\lvert 0\rangle_c + \lvert 1\rangle_c\bigr)\otimes\lvert 0\rangle_t.$$

The controlled block applies $\mathbb{1}$ to the target on the $c=0$ branch and
$R_x(2\pi) = -\mathbb{1}$ on the $c=1$ branch. The target itself is unchanged either way —
it is still $\lvert 0\rangle_t$, and it never becomes entangled with anything. But the $c=1$
branch has picked up a factor of $-1$:

$$\tfrac{1}{\sqrt2}\bigl(\lvert 0\rangle_c - \lvert 1\rangle_c\bigr)\otimes\lvert 0\rangle_t.$$

That is $\lvert -\rangle$, not $\lvert +\rangle$. The control has been flipped from one
equator point to the opposite one — by a rotation applied to a *different qubit*, which
returned that qubit to exactly where it started. The final Hadamard turns
$\lvert -\rangle$ into $\lvert 1\rangle$, and the measurement reads **1, every time**.

Do the same with $R_x(4\pi) = +\mathbb{1}$ and the control stays $\lvert +\rangle$, the final
Hadamard returns it to $\lvert 0\rangle$, and the measurement reads **0, every time**.

In [ ]:
def belt_trick(theta: float, seed: int = 17) -> tuple[Circuit, float]:
    """H-sandwich on a control, with an Rx(theta) applied to the target in between.

    Returns the circuit and the exact probability that the control reads 1.
    """
    qc_pair = Circuit(name="belt", seed=seed)
    c = qc_pair.alloc("c")
    t = qc_pair.alloc("t")
    H(c)                             # control into |+>: both branches now exist
    with qc_pair.control(c):
        Rx(t, theta=theta)           # turn the target, but only on the c = 1 branch
    H(c)                             # recombine the branches and read the phase
    # probabilities() is indexed by the basis state as an integer, qubit 0 (the control)
    # most significant. So indices 2 and 3 are |10> and |11>: the control reading 1.
    probs = qc_pair.inspect.probabilities()
    return qc_pair, float(probs[2] + probs[3])


def measure_control(theta: float, seed: int) -> int:
    """Run the experiment once from scratch and measure the control qubit for real."""
    fresh, _ = belt_trick(theta, seed=seed)
    return fresh.measure(fresh.qubits[0])   # qubits[0] is that circuit's own control


for label, theta in (("Rx(2π) = -1  (one turn) ", 2.0 * np.pi),
                     ("Rx(4π) = +1  (two turns)", 4.0 * np.pi),
                     ("no rotation at all      ", 0.0)):
    circuit, p_one = belt_trick(theta)
    # Five independent runs, five different seeds -- so a lucky random stream cannot be
    # what makes the answer come out the same each time.
    outcomes = [measure_control(theta, seed) for seed in range(5)]
    print(f"{label}  state {circuit.inspect.ket()!s:>16}   "
          f"P(control = 1) = {p_one:.12f}   five measurements: {outcomes}")

p_one_turn = belt_trick(2.0 * np.pi)[1]
p_two_turns = belt_trick(4.0 * np.pi)[1]

$P = 1$ against $P = 0$. Not a shifted distribution, not a statistical hint — two
deterministic, opposite answers, separated by nothing but whether the target qubit was
turned through $360°$ or $720°$.

Sit with what that means. The target qubit ends in exactly the state it started in. Its
Bloch vector never moved by the end; its density matrix is unchanged; every measurement you
could perform *on it* is unaffected. In the language of Part 5, the two experiments differ by
a rotation that is the **identity element of $SO(3)$**. And yet the control qubit comes out
in an orthogonal state, and a measurement tells you which one happened, every single shot.

**The $-1$ is physical.** Quaternion users carry it as bookkeeping — a sign that cancels in
the sandwich and is conventionally ignored, the leftover of a free choice about which side
of two mirrors to call the front. A qubit hands it to you as a measurement outcome.

The classical shadow of this is the **belt trick** (or plate trick, or Dirac scissors): hold
one end of a belt, rotate the other end through $360°$, and the belt is twisted; rotate
through another $360°$ in the *same* direction and, remarkably, the twist can be worked out
without turning either end again. The belt is tracking a path in rotation space, not just an
endpoint, and paths of length $2\pi$ and $4\pi$ are genuinely different. Objects that
transform this way are called **spinors**, and every electron, proton and neutron in your
body is one. Neutron interferometry measured exactly the experiment above in 1975 — the
$2\pi$-rotated beam came back out of phase with its unrotated twin, and the interference
fringes moved.

The punchline, restated: **the sign quaternions carry silently, a qubit can cash out.**

## Where to go next

- **[one_qubit_playground](one_qubit_playground.ipynb)** — the $R_x(2\pi) = -I$ observation
  this notebook set out to explain, plus the interferometer that turns any phase difference
  into a probability.
- **[01 — States and gates](../01-states-and-gates.ipynb)** — the Bloch sphere from scratch,
  if the geometry above went past too quickly.
- **[04 — Combinators](../04-combinators.ipynb)** — `with qc.control(c):` as a general tool,
  and why controlling a *block* rather than a gate is the interesting move; also
  `with qsim.within(V, q):`, the conjugation that Part 2's coda used to tilt a $\pi$-gate's
  axis. Phase kickback, which is what Part 6 is a minimal instance of, is the engine of
  phase estimation and therefore of Shor's algorithm.
- **[02 — Entanglement](../02-entanglement.ipynb)** — Part 6's controlled rotation left the
  two qubits *unentangled*, which is unusual and is exactly why the phase stayed readable.
  Entangling versions of the same move are where the Bloch vector starts to shrink.

## Assertions

Every claim above, re-checked numerically.

In [ ]:
# 1. The quaternion sandwich is the axis-angle rotation matrix (Part 1).
assert sandwich_error < 1e-12, sandwich_error

# 2. A reflection is an involution with determinant -1, and two of them make a rotation
#    by 2*alpha about the line where the mirrors meet (Part 2).
assert involution_error < 1e-12, involution_error
assert mirror_det_error < 1e-12, mirror_det_error
assert trace_angle_error < 1e-12, trace_angle_error
assert double_reflection_error < 1e-12, double_reflection_error
assert mirror_slide_error < 1e-12, mirror_slide_error
#    ...re-derived independently here on one fresh pair, angle checked to 1e-12.
check_n1 = np.array([0.0, 0.0, 1.0])
check_n2 = np.array([np.sin(0.4), 0.0, np.cos(0.4)])          # 0.4 rad from the first
check_m = reflection_matrix(check_n2) @ reflection_matrix(check_n1)
assert abs(np.linalg.det(reflection_matrix(check_n1)) + 1.0) < 1e-12
assert abs(np.linalg.det(check_m) - 1.0) < 1e-12
# arccos of (trace - 1) / 2 recovers the rotation angle whenever it is in [0, pi].
assert abs(float(np.arccos((np.trace(check_m) - 1.0) / 2.0)) - 0.8) < 1e-12
assert np.allclose(check_m, axis_angle_matrix(np.cross(check_n1, check_n2), 0.8), atol=1e-12)

# 3. A pure unit quaternion is a mirror: n v n reproduces reflect(v, n) (Part 2).
assert pure_sandwich_error < 1e-12, pure_sandwich_error

# 4. ...so q = n2 n1 is the rotation of the two mirrors, to the documented overall sign,
#    and flipping one normal negates q while changing neither mirror (Part 2).
assert mirror_quaternion_error < 1e-12, mirror_quaternion_error
assert mirror_parts_error < 1e-12, mirror_parts_error
assert sign_freedom_error < 1e-12, sign_freedom_error
check_q = hamilton(pure_quaternion(check_n2), pure_quaternion(check_n1))
assert np.allclose(check_q, -axis_angle_quaternion(np.cross(check_n1, check_n2), 0.8),
                   atol=1e-12), check_q
assert np.allclose(rotate_by_quaternion(check_q, np.array([0.3, -1.1, 0.7])),
                   check_m @ np.array([0.3, -1.1, 0.7]), atol=1e-12)

# 5. Every Hermitian-and-unitary gate is a mirror times the central inversion, and two of
#    them whose axes sit phi apart turn the Bloch vector by exactly 2*phi (Part 2's coda).
assert mirror_gate_error < 1e-12, mirror_gate_error
assert hxh_error < 1e-12, hxh_error
assert two_phi_error < 1e-10, two_phi_error

# 6. Every rotation gate maps to a UNIT quaternion whose components are exactly
#    (cos(theta/2), sin(theta/2) * axis) -- checked across three gates and six angles.
assert component_error < 1e-12, component_error
assert norm_error < 1e-12, norm_error
for rotation, unit_axis in gate_axes.values():
    for angle in [0.3, 1.0, np.pi, 2.5 * np.pi]:
        matrix = gate_matrix(lambda w, g=rotation, t=angle: g(w, theta=t))
        assert np.allclose(to_quaternion(matrix),
                           axis_angle_quaternion(unit_axis, angle), atol=1e-12)
        assert abs(quat_norm(to_quaternion(matrix)) - 1.0) < 1e-12

# 7. Matrix product <-> Hamilton product, over 300 random pairs (Part 3).
assert product_error < 1e-12, product_error

# 8. Quaternion conjugation rotates the Bloch vector exactly as the gates do (Part 4).
assert bloch_error < 1e-10, bloch_error

# 9. A full 2*pi turn is the quaternion -1: scalar part -1, vector part zero (Part 5).
full_turn = to_quaternion(gate_matrix(lambda w: Rx(w, theta=2 * np.pi)))
assert np.allclose(full_turn, [-1.0, 0.0, 0.0, 0.0], atol=1e-12), full_turn
double_turn = to_quaternion(gate_matrix(lambda w: Rx(w, theta=4 * np.pi)))
assert np.allclose(double_turn, [1.0, 0.0, 0.0, 0.0], atol=1e-12), double_turn
#    ...while the Bloch vector is 2*pi-periodic: the sweep's value at 2π equals its value at 0.
assert np.allclose(bloch_sweep[120], bloch_sweep[0], atol=1e-12)
assert np.allclose(bloch_sweep[240], bloch_sweep[0], atol=1e-12)

# 10. And that sign is an observable: the H-sandwich reads it out deterministically (Part 6).
assert abs(p_one_turn - 1.0) < 1e-12, p_one_turn
assert abs(p_two_turns - 0.0) < 1e-12, p_two_turns
assert abs(belt_trick(0.0)[1] - 0.0) < 1e-12

# 11. ...but on the qubit alone it is not: identical statistics, opposite state vectors.
assert turned_counts == still_counts
assert np.allclose(turned.inspect.state_vector(), -still.inspect.state_vector())

print("all assertions passed")